In [ ]:
# Imports and file paths
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# clear console
def clear_console():
    os.system('cls' if os.name == 'nt' else 'clear')
clear_console()

# load files (here: .tab or .txt files; you can add cores/records as you require)
files = {
    'core_name': 'core_file.tab',
    }

# read files and define columns. first column is age_ka, second is mg_ca or d18O
def read_two_column(filename):
    df = pd.read_csv(filename, sep='\t', comment='#', header=0)
    out = df.iloc[:, :2].copy()
    out.columns = ['age_ka', 'value']
    out['age_ka'] = pd.to_numeric(out['age_ka'], errors='coerce')   # errors='coerce' converts non-numeric values to NaN if neccessary
    out['value'] = pd.to_numeric(out['value'], errors='coerce')
    return out.dropna(subset=['age_ka', 'value']).reset_index(drop=True) # drop rows with NaNs in age_ka or value

# load files into a dict of dataframes
data = {}
for k, p in files.items():
    df = read_two_column(p)
    if k.endswith('MgCa'):
        df = df.rename(columns={'value': 'mg_ca'})
    else:
        df = df.rename(columns={'value': 'd18O'})
    data[k] = df

# d18O uncertainties (‰ VPDB)
d18O_sde_map = {
    'core_name': 0.03    # 0.03 is placeholder, please enter your uncertainty
    }


In [ ]:
# Calculation of temperature T from Mg/Ca and uncertainty propagation
def compute_temperature(df, core, sigma_calib=0.0):
    df = df.copy()
    mg = df['mg_ca'].astype(float)
    if core == 'core_name':               
        a = 0.1; sigma_a = 0.013            # exchange with values of the calibration, you want to use T=(mg-b)/a
        b = 1.0              
        sigma_mg = np.full_like(mg, 0.04, dtype=float)              # measurement uncertainty of Mg/Ca
    else:
        raise ValueError(core)

    T = (mg - b) / a
    # propagate uncertainties:
    term1 = (sigma_mg / a) ** 2
    term2 = (((mg - b) / (a ** 2)) * sigma_a) ** 2
    # include calibration uncertainty (sigma_calib) in quadrature if given
    sigma_T = np.sqrt(term1 + term2 + sigma_calib**2)

    df['T_C'] = T
    df['T_uncertainty'] = sigma_T

    # return the full DataFrame and reset the row index
    return df.reset_index(drop=True)

# apply to Mg/Ca dataframe, include calibration uncertainty
mg_dfs = {}
mg_dfs['core_name'] = compute_temperature(data['core_name_MgCa'], 'core_name', sigma_calib=0.8) #adapt sigma_calib with your unique calibration uncertainty

# show brief previews
for core, df in mg_dfs.items():
    print('\n---', core, 'rows:', len(df))
    display(df.head())

In [ ]:
# generalized least squares interpolation
def acvf(x, var=1, corrlength=2, acvftype='G'):
    x = np.asarray(x)
    if acvftype == 'E':
        exparg = -(1.0 / corrlength) * np.abs(x)
    elif acvftype == 'G':
        exparg = -(1.0 / (2.0 * corrlength ** 2)) * x ** 2
    else:
        raise ValueError(f"acvf: priortype {acvftype} unknown")
    return var * np.exp(exparg)

def gls(xd, d, sde, xm, var, corrl, acvftype='G'):
    Sigma_mm = acvf(xm[:, None] - xm[None, :], var, corrl, acvftype)
    A = acvf(xm[:, None] - xd[None, :], var, corrl, acvftype)
    B = acvf(xd[:, None] - xd[None, :], var, corrl, acvftype)
    R = np.diag(sde ** 2)
    K = B + R
    alpha = np.linalg.solve(K, d)
    m_post = A @ alpha
    V = Sigma_mm - A @ np.linalg.solve(K, A.T)
    post_std = np.sqrt(np.maximum(np.diag(V), 0.0))
    return m_post, post_std

# initialize storage for core GLS outputs 
core_grid = {}
gz_xm = {}
gz_d18Osw = {}
gz_unc = {}

for core in ('core_name'):
    # load data for core
    td = mg_dfs[core].copy()
    xd_T = td['age_ka'].values
    d_T = td['T_C'].values
    sde_T = td['T_uncertainty'].values
    od = data[core + '_d18O'].copy()
    xd_O = od['age_ka'].values
    d_O = od['d18O'].values
    sde_O = np.full_like(d_O, d18O_sde_map[core], dtype=float)

    # per-core grid point counts to ensure grid spacing << correlation length
    n_map = {'core_name': 500}            # adapt n_map with your desired grid point counts (e.g., 500 points for 360-450 ka gives ~0.18 ka spacing)
    npts = n_map.get(core, 500)           # default to 500 if core not in n_map
    xm = np.linspace(min(xd_T.min(), xd_O.min()), max(xd_T.max(), xd_O.max()), npts)

    # set prior variance and correlation length per dataset
    var_T = float(np.var(d_T))
    corrl_T = float(2*np.mean(np.diff(np.sort(xd_T))))  # correlation length ~2x mean age spacing of data points (adapt if T is more/less variable than d18O)
    var_O = float(np.var(d_O))
    corrl_O = float(2.5*np.mean(np.diff(np.sort(xd_O))))    # correlation length ~2.5x mean age spacing of data points (d18O may be more variable than T, so slightly longer correlation length can help smooth noise)

    # GLS (Gaussian ACVF)
    m_T, std_T = gls(xd_T, d_T, sde_T, xm, var_T, corrl_T, acvftype='G')
    m_O, std_O = gls(xd_O, d_O, sde_O, xm, var_O, corrl_O, acvftype='G')

    # for display: age minimum - age maximum ka
    display_mask = (xm >= 360.0) & (xm <= 450) # adapt age limits for display as needed
    xm_disp = xm[display_mask]
    m_T_disp = m_T[display_mask]
    std_T_disp = std_T[display_mask]
    m_O_disp = m_O[display_mask]
    std_O_disp = std_O[display_mask]

    # compute d18Osw
    d18Osw = (m_O_disp + 0.27) - 0.25 * (16.9 - m_T_disp)              # example Mg/Ca-T-d18Osw relationship, adapt as needed for your specific calibration
    combined_std = np.sqrt(std_O_disp**2 + (0.25*std_T_disp)**2)       # propagate uncertainties from T and d18O to d18Osw

    # save core GLS outputs (ages, mean d18Osw, uncertainty)
    core_grid[core] = {'age': xm_disp, 'd18Osw': d18Osw, 'unc': combined_std}
    gz_xm[core] = xm_disp
    gz_d18Osw[core] = d18Osw
    gz_unc[core] = combined_std

In [ ]:
# read ice-volume d18O reconstruction(s)

files = {
    'd18Oice_name': 'd18Oice_file.xlsx' # contains age_ka, d18Oice, and available uncertainty columns; adapt with your file and column names as needed
}

dfs = {}
for name, path in files.items():
    try:
        df = pd.read_excel(path, header=None)
    except Exception as e:
        print(f'Could not read {path}: {e}')
        continue
    df = df.iloc[:, :3].copy()
    cols = ['age_ka', 'd18Oice', 'uncertainty'][:df.shape[1]] # assign column names 
    df.columns = cols
    df['age_ka'] = pd.to_numeric(df['age_ka'], errors='coerce')
    df['d18Oice'] = pd.to_numeric(df['d18Oice'], errors='coerce')
    if 'uncertainty' in df.columns:
        df['uncertainty'] = pd.to_numeric(df['uncertainty'], errors='coerce')
    else:
        df['uncertainty'] = np.nan
    dfs[name] = df.dropna(subset=['age_ka', 'd18Oice']).reset_index(drop=True) 


In [ ]:
# GLS-fit ice reconstruction onto core's saved GLS grid,
# save results in `ice_on_core` and compute d180w-ice (here: "local"): `d18Olocal` = core_d18Osw - ice_on_core
ice_on_core = {}
d18Olocal = {}
for core in ('core_name'):
    xm_core = gz_xm[core]            # ages saved from core GLS
    core_mean = gz_d18Osw[core]
    core_unc  = gz_unc[core]
    ice_on_core[core] = {}
    d18Olocal[core] = {}
    for name, df in dfs.items():
        df = df.sort_values('age_ka')
        xd = df['age_ka'].values
        d = df['d18Oice'].values
        if 'uncertainty' in df.columns:
            sde = df['uncertainty'].values.astype(float)
        else:
            sde = np.full_like(d, np.nan, dtype=float)
        if np.all(np.isnan(sde)):
            sde = np.full_like(d, 0.05, dtype=float)  # assigns a default uncertainty of 0.05 ‰ if no uncertainty information is available
        else:
            nanmask = np.isnan(sde)
            if nanmask.any():
                if np.any(~nanmask):
                    sde[nanmask] = np.nanmedian(sde[~nanmask])
                else:
                    sde[nanmask] = 0.05 # if all values are NaN, assign default uncertainty
        # set prior variance & correlation length for the ice record
        var_ice = float(np.var(d)) if len(d) > 0 else 1.0
        corrl_ice = float(2.0 * np.mean(np.diff(np.sort(xd)))) if len(xd) > 1 else 2.0 # correlation length ~2x mean age spacing of ice reconstruction data points (adapt if your ice record is more/less variable)
        # GLS (Gaussian ACVF) to predict ice record on core grid
        m_ice, std_ice = gls(xd, d, sde, xm_core, var_ice, corrl_ice, acvftype='G')
        ice_on_core[core][name] = {'age': xm_core, 'mean': m_ice, 'unc': std_ice}
        
        # compute local d18O = core_d18Osw - ice_on_core and propagate uncertainties
        local_mean = core_mean - m_ice
        local_unc = np.sqrt(core_unc**2 + std_ice**2)
        d18Olocal[core][name] = {'age': xm_core, 'mean': local_mean, 'unc': local_unc}


In [ ]:
# Export final Excel files per core and combined d18Oice inputs
import os
import pandas as pd
import numpy as np

outdir = 'results'
os.makedirs(outdir, exist_ok=True)

for core in ('core_name'):
    # Temperature (from Mg/Ca) with uncertainties
    td = mg_dfs.get(core).copy()
    df_temp = td.rename(columns={'age_ka': 'age_ka', 'T_C': 'temperature_C', 'T_uncertainty': 'temperature_uncertainty_C'})

    # Original measured d18O (tabular input)
    ddf = data.get(core + '_d18O').copy()
    ddf = ddf.rename(columns={'age_ka': 'age_ka', 'd18O': 'd18O_measured_VPDB'})

    # GLS-derived d18Osw on the core display grid
    age = np.asarray(gz_xm[core])
    d18Osw = np.asarray(gz_d18Osw[core])
    unc = np.asarray(gz_unc[core])
    df_sw = pd.DataFrame({'age_ka': age, 'd18Osw_permil': d18Osw, 'd18Osw_uncertainty_permil': unc})

    # Write to Excel with separate sheets
    outpath = os.path.join(outdir, f'{core}_final.xlsx')
    try:
        with pd.ExcelWriter(outpath, engine='openpyxl') as writer:
            df_temp.to_excel(writer, sheet_name='Temperature', index=False)
            ddf.to_excel(writer, sheet_name='d18O_measured', index=False)
            df_sw.to_excel(writer, sheet_name='d18Osw', index=False)

            # d18O_local for each ice reconstruction and ΔBWS conversion
            for name in ('d18Oice_name'):
                rec = d18Olocal.get(core, {}).get(name, None)
                if rec is None:
                    continue
                df_local = pd.DataFrame({'age_ka': rec['age'], 'd18O_local_permil': rec['mean'], 'd18O_local_uncertainty_permil': rec['unc']})
                # convert to ΔBWS (psu) using factor 0.24±0.014 ‰ per psu (adapt as needed for your specific calibration!) and propagate uncertainty
                k = 0.24    # exchange with your specific d18O-ΔBWS conversion factor
                sigma_k = 0.014 
                # Calculate DeltaBWS
                df_local['DeltaBWS_psu'] = df_local['d18O_local_permil'] / k
                # Propagate uncertainty (both numerator and denominator)
                df_local['DeltaBWS_uncertainty_psu'] = abs(df_local['DeltaBWS_psu']) * (
                    ((df_local['d18O_local_uncertainty_permil'] / df_local['d18O_local_permil'])**2 + (sigma_k / k)**2) ** 0.5)
                df_local.to_excel(writer, sheet_name=f'd18Olocal_{name}', index=False)
    except Exception as e:
        print('Could not write', outpath, e)
    else:
        print('Wrote', outpath)

# Combined d18Oice input workbook
outpath_inputs = os.path.join(outdir, 'd18Oice_inputs.xlsx')
try:
    with pd.ExcelWriter(outpath_inputs, engine='openpyxl') as writer:
        for name, df in dfs.items():
            dfi = df.copy()
            dfi.to_excel(writer, sheet_name=name, index=False)
except Exception as e:
    print('Could not write', outpath_inputs, e)
else:
    print('Wrote', outpath_inputs)